
# FAIR Universe Weak Lensing Challenge — Research-Guided Advanced Notebook

This notebook blends a literature-grounded review with an end-to-end pipeline for Phase 1 of the FAIR Universe Weak Lensing ML Uncertainty Challenge. It expands upon the provided starting kits by integrating recent research insights, advanced modeling strategies, and reproducible utilities so you can train a competitive submission-ready model.



## How to use this notebook

1. **Research-first:** automatically query astrophysics literature (arXiv) to understand successful approaches to weak-lensing parameter inference.
2. **Data preparation:** leverage configurable helpers for loading, noising, splitting, and visualizing the convergence maps.
3. **Modeling:** train an enhanced residual CNN with heteroscedastic uncertainty heads, mixup augmentation, gradient clipping, and early stopping.
4. **Evaluation & inference:** compute the official Phase 1 score, validate calibration, and prepare Codabench submissions.

> **Tip:** Run cells in order. Cells that train models are intentionally modular—feel free to tweak hyperparameters or plug in your own architectures.



## Notebook outline

- [0. Environment setup](#0---Environment-setup)
- [1. Literature review by programmatic paper search](#1---Literature-review-by-programmatic-paper-search)
- [2. Data handling utilities](#2---Data-handling-utilities)
- [3. Exploratory analysis & visualization](#3---Exploratory-analysis--visualization)
- [4. Research-inspired advanced CNN model](#4---Research-inspired-advanced-CNN-model)
- [5. Training & validation loop](#5---Training--validation-loop)
- [6. Diagnostics & calibration](#6---Diagnostics--calibration)
- [7. Test-time inference & submission packaging](#7---Test-time-inference--submission-packaging)
- [8. Next steps](#8---Next-steps)


---
### 0 - Environment setup


The cell below mirrors the starter notebooks by detecting Google Colab and cloning the repository if needed. When running locally (e.g., inside this repository), it simply does nothing.

In [ ]:

COLAB = 'google.colab' in str(get_ipython())
if COLAB:
    !git clone --depth 1 https://github.com/FAIR-Universe/Cosmology_Challenge.git
    %cd Cosmology_Challenge


#### Core imports & global settings

In [ ]:

import os
import json
import time
import copy
import math
import zipfile
import datetime
import warnings
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import requests
import xml.etree.ElementTree as ET

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')


In [ ]:

# Ensure reproducibility across libraries

def set_global_seed(seed: int = 314159) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_global_seed()


---
### 1 - Literature review by programmatic paper search


A fast way to keep up with the state of the art is to query **arXiv** for papers related to weak-lensing inference and deep learning. The helper below wraps the Atom API so you can adapt the keywords or dig deeper.

In [ ]:

ARXIV_NAMESPACE = '{http://www.w3.org/2005/Atom}'

def query_arxiv(search_terms: str, max_results: int = 5, sort_by: str = 'relevance') -> pd.DataFrame:
    """Query the arXiv API and return a tidy DataFrame with key fields."""
    params = {
        'search_query': f'all:{search_terms}',
        'start': 0,
        'max_results': max_results,
        'sortBy': sort_by,
        'sortOrder': 'descending'
    }
    url = 'http://export.arxiv.org/api/query'
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    root = ET.fromstring(response.text)
    records: List[Dict[str, str]] = []
    for entry in root.findall(f'{ARXIV_NAMESPACE}entry'):
        title = entry.find(f'{ARXIV_NAMESPACE}title').text.strip().replace('
', ' ')
        summary = entry.find(f'{ARXIV_NAMESPACE}summary').text.strip().replace('
', ' ')
        authors = ', '.join(author.find(f'{ARXIV_NAMESPACE}name').text for author in entry.findall(f'{ARXIV_NAMESPACE}author'))
        published = entry.find(f'{ARXIV_NAMESPACE}published').text[:10]
        link = entry.find(f'{ARXIV_NAMESPACE}id').text
        records.append({
            'title': title,
            'authors': authors,
            'published': published,
            'link': link,
            'summary': summary
        })

    return pd.DataFrame(records)


In [ ]:

search_topics = {
    'CNN based weak-lensing inference': 'weak lensing convolutional neural network cosmology',
    'Uncertainty-aware weak-lensing': 'weak lensing Bayesian deep learning uncertainty',
}

literature_results = {
    topic: query_arxiv(query, max_results=5)
    for topic, query in search_topics.items()
}

for topic, df in literature_results.items():
    print(f"
=== {topic} ===")
    display(df[['published', 'title', 'authors', 'link']])



#### Key takeaways from recent literature

- **Deep CNN mass-map inference** (e.g., Ribli *et al.*, 2019; Peel *et al.*, 2019) highlights the benefit of residual and U-Net style skip connections to better capture non-Gaussian statistics in convergence maps.
- **Simulation-based inference (SBI)** approaches such as neural density estimators (Alsing *et al.*, 2019) and normalizing flows motivate predicting *distributional* outputs instead of single point estimates.
- **Uncertainty calibration** is critical. Works like Jeffrey *et al.* (2021) and Adam *et al.* (2022) combine Bayesian neural networks with strong data augmentation to yield realistic credible intervals.
- **Physics-informed augmentations** (rotations, parity flips, shape noise realizations) are repeatedly emphasized as cheap ways to enlarge training sets while respecting symmetries of weak-lensing maps.

The model we build below incorporates these lessons: residual blocks with squeeze-excitation for richer receptive fields, heteroscedastic Gaussian heads for predictive uncertainties, rotation/flip augmentations, and coverage diagnostics to monitor calibration.


---
### 2 - Data handling utilities


In [ ]:

class Utility:
    """Helper collection for noise injection, persistence, and splits."""

    @staticmethod
    def add_noise(data: np.ndarray, mask: np.ndarray, ng: float, pixel_size: float = 2.0) -> np.ndarray:
        noise = np.random.randn(*data.shape) * 0.4 / (2 * ng * pixel_size**2) ** 0.5
        return data + noise * mask

    @staticmethod
    def load_np(data_dir: str, file_name: str) -> np.ndarray:
        file_path = os.path.join(data_dir, file_name)
        return np.load(file_path)

    @staticmethod
    def save_np(data_dir: str, file_name: str, data: np.ndarray) -> None:
        os.makedirs(data_dir, exist_ok=True)
        file_path = os.path.join(data_dir, file_name)
        np.save(file_path, data)

    @staticmethod
    def save_json_zip(submission_dir: str, json_file_name: str, zip_file_name: str, data: Dict[str, np.ndarray]) -> str:
        os.makedirs(submission_dir, exist_ok=True)
        json_path = os.path.join(submission_dir, json_file_name)
        with open(json_path, 'w') as f:
            json.dump(data, f)
        zip_path = os.path.join(submission_dir, zip_file_name)
        with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
            zf.write(json_path, arcname=json_file_name)
        os.remove(json_path)
        return zip_path

    @staticmethod
    def train_val_split(num_realizations: int, val_fraction: float = 0.2, seed: int = 314159) -> Tuple[np.ndarray, np.ndarray]:
        rng = np.random.default_rng(seed)
        indices = np.arange(num_realizations)
        rng.shuffle(indices)
        val_size = max(1, int(len(indices) * val_fraction))
        val_idx = indices[:val_size]
        train_idx = indices[val_size:]
        return train_idx, val_idx

    @staticmethod
    def compute_image_stats(images: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        mean = images.mean(axis=(0, 1, 2))
        std = images.std(axis=(0, 1, 2)) + 1e-6
        return mean.astype(np.float32), std.astype(np.float32)


In [ ]:

class Data:
    def __init__(self, data_dir: str, use_public_dataset: bool = False):
        self.data_dir = data_dir
        self.use_public_dataset = use_public_dataset
        self.mask_file = 'WIDE12H_bin2_2arcmin_mask.npy'
        self.viz_label_file = 'label.npy'

        if self.use_public_dataset:
            self.kappa_file = 'WIDE12H_bin2_2arcmin_kappa.npy'
            self.label_file = self.viz_label_file
            self.test_kappa_file = 'WIDE12H_bin2_2arcmin_kappa_noisy_test.npy'
            self.Ncosmo = 101
            self.Nsys = 256
            self.Ntest = 4000
        else:
            self.kappa_file = 'sampled_WIDE12H_bin2_2arcmin_kappa.npy'
            self.label_file = 'sampled_label.npy'
            self.test_kappa_file = 'sampled_WIDE12H_bin2_2arcmin_kappa_noisy_test.npy'
            self.Ncosmo = 3
            self.Nsys = 30
            self.Ntest = 3

        self.shape = (1424, 176)
        self.pixel_size_arcmin = 2.0
        self.pixel_size_radian = self.pixel_size_arcmin / 60 / 180 * np.pi
        self.ng = 30

    def load_train_data(self) -> None:
        self.mask = Utility.load_np(self.data_dir, self.mask_file)
        raw_kappa = Utility.load_np(self.data_dir, self.kappa_file)
        self.kappa = np.zeros((self.Ncosmo, self.Nsys, *self.shape), dtype=np.float32)
        self.kappa[:, :, self.mask] = raw_kappa
        self.label = Utility.load_np(self.data_dir, self.label_file).astype(np.float32)
        self.viz_label = Utility.load_np(self.data_dir, self.viz_label_file).astype(np.float32)

    def load_test_data(self) -> None:
        raw_test = Utility.load_np(self.data_dir, self.test_kappa_file)
        self.kappa_test = np.zeros((self.Ntest, *self.shape), dtype=np.float32)
        self.kappa_test[:, self.mask] = raw_test


#### Load dataset

In [ ]:

root_dir = os.getcwd()
DATA_DIR = os.path.join(root_dir, 'input_data')
USE_PUBLIC_DATASET = False  # Switch to True after downloading the full dataset locally
PUBLIC_DATA_DIR = '[DEFINE PATH]'  # Only used when USE_PUBLIC_DATASET = True

data_dir = PUBLIC_DATA_DIR if USE_PUBLIC_DATASET else DATA_DIR

data_obj = Data(data_dir=data_dir, use_public_dataset=USE_PUBLIC_DATASET)
data_obj.load_train_data()
data_obj.load_test_data()

print(f"Number of cosmologies: {data_obj.Ncosmo}")
print(f"Number of systematic realizations per cosmology: {data_obj.Nsys}")
print(f"Train kappa shape: {data_obj.kappa.shape}")
print(f"Train labels shape: {data_obj.label.shape}")
print(f"Mask coverage: {data_obj.mask.mean():.4f}")
print(f"Test kappa shape: {data_obj.kappa_test.shape}")


#### Add survey noise, reshape, and split

In [ ]:

noisy_kappa = Utility.add_noise(data_obj.kappa, data_obj.mask, data_obj.ng, pixel_size=data_obj.pixel_size_arcmin)

N_cosmo, N_sys = noisy_kappa.shape[:2]
image_shape = data_obj.shape

X = noisy_kappa.reshape(N_cosmo * N_sys, *image_shape)
y_full = data_obj.label.reshape(N_cosmo * N_sys, -1)
y = y_full[:, :2]

train_idx, val_idx = Utility.train_val_split(num_realizations=N_sys, val_fraction=0.2)
train_indices = np.concatenate([train_idx + i * N_sys for i in range(N_cosmo)])
val_indices = np.concatenate([val_idx + i * N_sys for i in range(N_cosmo)])

X_train, X_val = X[train_indices], X[val_indices]
y_train, y_val = y[train_indices], y[val_indices]

print(f"Training set: {X_train.shape}, Validation set: {X_val.shape}")


In [ ]:

image_mean, image_std = Utility.compute_image_stats(X_train)
label_min = y_train.min(axis=0)
label_max = y_train.max(axis=0)

print(f"Training image mean: {image_mean}")
print(f"Training image std: {image_std}")
print(f"Parameter min: {label_min}")
print(f"Parameter max: {label_max}")


---
### 3 - Exploratory analysis & visualization


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(data_obj.mask, cmap='magma')
axes[0].set_title('Survey mask')
axes[0].axis('off')

axes[1].imshow(data_obj.kappa[0, 0], cmap='viridis')
axes[1].set_title('Example noiseless map')
axes[1].axis('off')

axes[2].imshow(noisy_kappa[0, 0], cmap='viridis')
axes[2].set_title('Example noisy map')
axes[2].axis('off')
plt.show()


In [ ]:

cosmo_df = pd.DataFrame(
    data_obj.viz_label.reshape(-1, data_obj.viz_label.shape[-1]),
    columns=['Omega_m', 'S8', 'A_bary', 'eta0', 'DeltaZ']
)

sns.pairplot(cosmo_df[['Omega_m', 'S8']])
plt.suptitle('Cosmological parameter distribution (full training set)', y=1.02)
plt.show()


---
### 4 - Research-inspired advanced CNN model


In [ ]:

@dataclass
class Config:
    IMG_HEIGHT: int = image_shape[0]
    IMG_WIDTH: int = image_shape[1]
    DEVICE: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    BATCH_SIZE: int = 12
    VAL_BATCH_SIZE: int = 24
    NUM_EPOCHS: int = 60
    LEARNING_RATE: float = 3e-4
    WEIGHT_DECAY: float = 1e-4
    MIXUP_ALPHA: float = 0.3
    GRAD_CLIP_NORM: float = 1.0
    EARLY_STOPPING_PATIENCE: int = 12
    EARLY_STOPPING_DELTA: float = 1e-4
    LOG_VAR_REG: float = 1e-4
    USE_AMP: bool = True
    LABEL_NAMES: Tuple[str, str] = ('Omega_m', 'S8')

config = Config()
print(f"Using device: {config.DEVICE}")


In [ ]:

class CosmologyDataset(Dataset):
    def __init__(self, images: np.ndarray, labels: np.ndarray, mean: np.ndarray, std: np.ndarray, augment: bool = False):
        self.images = images.astype(np.float32)
        self.labels = labels.astype(np.float32)
        self.mean = mean.reshape(1, 1, 1).astype(np.float32)
        self.std = std.reshape(1, 1, 1).astype(np.float32)
        self.augment = augment

    def __len__(self) -> int:
        return len(self.images)

    @staticmethod
    def random_augment(image: np.ndarray) -> np.ndarray:
        if np.random.rand() < 0.5:
            image = np.flip(image, axis=1)
        if np.random.rand() < 0.5:
            image = np.flip(image, axis=0)
        k = np.random.randint(0, 4)
        image = np.rot90(image, k)
        return image.copy()

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        image = self.images[idx]
        if self.augment:
            image = self.random_augment(image)
        image = (image - self.mean) / self.std
        image = torch.from_numpy(image).unsqueeze(0)
        label = torch.from_numpy(self.labels[idx])
        return image, label


In [ ]:

train_dataset = CosmologyDataset(X_train, y_train, image_mean, image_std, augment=True)
val_dataset = CosmologyDataset(X_val, y_val, image_mean, image_std, augment=False)

train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=0, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=config.VAL_BATCH_SIZE, shuffle=False, num_workers=0)


In [ ]:

class SqueezeExcitation(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, kernel_size=1),
            nn.SiLU(),
            nn.Conv2d(channels // reduction, channels, kernel_size=1),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        scale = self.pool(x)
        scale = self.fc(scale)
        return x * scale


class ResidualBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1, use_se: bool = True):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.se = SqueezeExcitation(out_channels) if use_se else nn.Identity()
        self.activation = nn.SiLU()

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = self.shortcut(x)
        out = self.activation(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        out += identity
        out = self.activation(out)
        return out


class CosmologyRegressor(nn.Module):
    def __init__(self, label_min: np.ndarray, label_max: np.ndarray):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(32),
            nn.SiLU(),
        )

        self.layer1 = nn.Sequential(
            ResidualBlock(32, 64, stride=2),
            ResidualBlock(64, 64)
        )
        self.layer2 = nn.Sequential(
            ResidualBlock(64, 128, stride=2),
            ResidualBlock(128, 128)
        )
        self.layer3 = nn.Sequential(
            ResidualBlock(128, 256, stride=2),
            ResidualBlock(256, 256)
        )
        self.layer4 = nn.Sequential(
            ResidualBlock(256, 512, stride=2),
            ResidualBlock(512, 512)
        )

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.backbone_dropout = nn.Dropout(0.2)
        self.backbone_fc = nn.Sequential(
            nn.Linear(512, 256),
            nn.SiLU(),
            nn.Dropout(0.2),
        )
        self.mean_head = nn.Linear(256, 2)
        self.log_var_head = nn.Linear(256, 2)

        self.register_buffer('label_min', torch.from_numpy(label_min.astype(np.float32)))
        self.register_buffer('label_max', torch.from_numpy(label_max.astype(np.float32)))
        self.log_var_clip = (-5.0, 4.0)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x).flatten(1)
        x = self.backbone_dropout(x)
        x = self.backbone_fc(x)
        mean_raw = self.mean_head(x)
        mean = torch.sigmoid(mean_raw) * (self.label_max - self.label_min) + self.label_min
        log_var = self.log_var_head(x)
        log_var = torch.clamp(log_var, *self.log_var_clip)
        return mean, log_var


---
### 5 - Training & validation loop


In [ ]:

def gaussian_nll(mean: torch.Tensor, log_var: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    inv_var = torch.exp(-log_var)
    nll = 0.5 * (log_var + (target - mean) ** 2 * inv_var)
    return nll.mean()


def heteroscedastic_loss(mean: torch.Tensor, log_var: torch.Tensor, target: torch.Tensor, log_var_reg: float = 0.0) -> torch.Tensor:
    loss = gaussian_nll(mean, log_var, target)
    if log_var_reg > 0:
        loss += log_var_reg * (log_var ** 2).mean()
    return loss


class EarlyStopping:
    def __init__(self, patience: int = 10, min_delta: float = 1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best = math.inf
        self.counter = 0

    def step(self, metric: float) -> bool:
        if metric + self.min_delta < self.best:
            self.best = metric
            self.counter = 0
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience


In [ ]:

def train_one_epoch(model: nn.Module, dataloader: DataLoader, optimizer: torch.optim.Optimizer,
                    scaler: torch.cuda.amp.GradScaler, device: str, mixup_alpha: float, log_var_reg: float) -> float:
    model.train()
    running_loss = 0.0
    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        if mixup_alpha > 0:
            lam = np.random.beta(mixup_alpha, mixup_alpha)
            index = torch.randperm(images.size(0), device=device)
            images = lam * images + (1 - lam) * images[index]
            labels = lam * labels + (1 - lam) * labels[index]

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=config.USE_AMP):
            mean, log_var = model(images)
            loss = heteroscedastic_loss(mean, log_var, labels, log_var_reg=log_var_reg)
        scaler.scale(loss).backward()
        nn.utils.clip_grad_norm_(model.parameters(), config.GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)

    return running_loss / len(dataloader.dataset)


def evaluate_model(model: nn.Module, dataloader: DataLoader, device: str, log_var_reg: float) -> Tuple[float, Dict[str, np.ndarray]]:
    model.eval()
    running_loss = 0.0
    preds_mean, preds_log_var, truths = [], [], []
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)
            mean, log_var = model(images)
            loss = heteroscedastic_loss(mean, log_var, labels, log_var_reg=log_var_reg)
            running_loss += loss.item() * images.size(0)
            preds_mean.append(mean.cpu().numpy())
            preds_log_var.append(log_var.cpu().numpy())
            truths.append(labels.cpu().numpy())

    mean_array = np.concatenate(preds_mean)
    log_var_array = np.concatenate(preds_log_var)
    truth_array = np.concatenate(truths)

    return running_loss / len(dataloader.dataset), {
        'mean': mean_array,
        'log_var': log_var_array,
        'truth': truth_array,
    }


In [ ]:

def compute_metrics_from_outputs(outputs: Dict[str, np.ndarray]) -> Dict[str, float]:
    mean = outputs['mean']
    truth = outputs['truth']
    std = np.exp(0.5 * outputs['log_var'])
    mse = np.mean((truth - mean) ** 2, axis=0)
    mae = np.mean(np.abs(truth - mean), axis=0)
    rmse = np.sqrt(mse)
    coverage_68 = np.mean(np.abs(truth - mean) <= std, axis=0)
    coverage_95 = np.mean(np.abs(truth - mean) <= 2 * std, axis=0)
    return {
        'mae_Omega_m': mae[0],
        'mae_S8': mae[1],
        'rmse_Omega_m': rmse[0],
        'rmse_S8': rmse[1],
        'coverage68_Omega_m': coverage_68[0],
        'coverage68_S8': coverage_68[1],
        'coverage95_Omega_m': coverage_95[0],
        'coverage95_S8': coverage_95[1],
    }


In [ ]:

class Score:
    @staticmethod
    def _score_phase1(true_cosmo: np.ndarray, infer_cosmo: np.ndarray, errorbar: np.ndarray) -> float:
        sq_error = (true_cosmo - infer_cosmo) ** 2
        scale_factor = 1000
        score = - np.sum(sq_error / errorbar ** 2 + np.log(errorbar ** 2) + scale_factor * sq_error, axis=1)
        score = np.mean(score)
        return score if score >= -1e6 else -1e6


In [ ]:

def train_model(train_loader: DataLoader, val_loader: DataLoader, config: Config,
                label_min: np.ndarray, label_max: np.ndarray,
                num_epochs: Optional[int] = None) -> Tuple[CosmologyRegressor, Dict[str, List[float]], Dict[str, np.ndarray]]:
    device = config.DEVICE
    model = CosmologyRegressor(label_min=label_min, label_max=label_max).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=4, verbose=True)
    scaler = torch.cuda.amp.GradScaler(enabled=config.USE_AMP)
    stopper = EarlyStopping(patience=config.EARLY_STOPPING_PATIENCE, min_delta=config.EARLY_STOPPING_DELTA)

    history = {'train_loss': [], 'val_loss': [], 'val_score': []}
    best_state = copy.deepcopy(model.state_dict())
    best_outputs = None
    best_val_loss = math.inf

    epochs = num_epochs or config.NUM_EPOCHS
    for epoch in range(1, epochs + 1):
        start = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, scaler, device,
                                     mixup_alpha=config.MIXUP_ALPHA, log_var_reg=config.LOG_VAR_REG)
        val_loss, val_outputs = evaluate_model(model, val_loader, device, log_var_reg=config.LOG_VAR_REG)
        val_std = np.exp(0.5 * val_outputs['log_var'])
        val_score = Score._score_phase1(val_outputs['truth'], val_outputs['mean'], val_std)
        scheduler.step(val_loss)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_score'].append(val_score)

        elapsed = time.time() - start
        print(f"Epoch {epoch:02d} | Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f} | Val score: {val_score:.3f} | {elapsed:.1f}s")

        if val_loss < best_val_loss - config.EARLY_STOPPING_DELTA:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            best_outputs = val_outputs

        if stopper.step(val_loss):
            print('Early stopping triggered.')
            break

    model.load_state_dict(best_state)
    return model, history, best_outputs


#### Train (or load) the model

In [ ]:

MODEL_DIR = os.path.join(root_dir, 'artifacts', 'phase1_advanced_model')
os.makedirs(MODEL_DIR, exist_ok=True)
MODEL_PATH = os.path.join(MODEL_DIR, 'cosmology_regressor.pt')
HISTORY_PATH = os.path.join(MODEL_DIR, 'training_history.json')

USE_PRETRAINED_MODEL = False  # Switch to True after training to reuse saved weights

if not USE_PRETRAINED_MODEL:
    trained_model, history, best_outputs = train_model(train_loader, val_loader, config, label_min, label_max)
    torch.save({'state_dict': trained_model.state_dict(), 'label_min': label_min, 'label_max': label_max}, MODEL_PATH)
    with open(HISTORY_PATH, 'w') as f:
        json.dump(history, f, indent=2)
else:
    trained_model = CosmologyRegressor(label_min=label_min, label_max=label_max).to(config.DEVICE)
    checkpoint = torch.load(MODEL_PATH, map_location=config.DEVICE)
    trained_model.load_state_dict(checkpoint['state_dict'])
    with open(HISTORY_PATH, 'r') as f:
        history = json.load(f)
    _, _, best_outputs = evaluate_model(trained_model, val_loader, config.DEVICE, log_var_reg=config.LOG_VAR_REG)


---
### 6 - Diagnostics & calibration


In [ ]:

if 'history' in globals():
    epochs_range = range(1, len(history['train_loss']) + 1)
    fig, ax1 = plt.subplots(figsize=(10, 4))
    ax1.plot(epochs_range, history['train_loss'], label='Train loss')
    ax1.plot(epochs_range, history['val_loss'], label='Val loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()

    ax2 = ax1.twinx()
    ax2.plot(epochs_range, history['val_score'], color='green', label='Val score')
    ax2.set_ylabel('Phase 1 score')
    ax2.legend(loc='lower right')
    plt.title('Training dynamics')
    plt.show()


In [ ]:

val_means = best_outputs['mean']
val_truth = best_outputs['truth']
val_stds = np.exp(0.5 * best_outputs['log_var'])

metrics = compute_metrics_from_outputs(best_outputs)
print('Validation metrics:')
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for i, ax in enumerate(axes):
    ax.errorbar(val_truth[:, i], val_means[:, i], yerr=val_stds[:, i], fmt='o', alpha=0.5)
    ax.plot([val_truth[:, i].min(), val_truth[:, i].max()], [val_truth[:, i].min(), val_truth[:, i].max()], 'k--')
    ax.set_xlabel('Ground truth')
    ax.set_ylabel('Prediction')
    ax.set_title(config.LABEL_NAMES[i])
plt.tight_layout()
plt.show()


---
### 7 - Test-time inference & submission packaging


In [ ]:

trained_model.eval()
with torch.no_grad():
    val_score = Score._score_phase1(val_truth, val_means, val_stds)
print(f"Final validation Phase 1 score: {val_score:.3f}")


In [ ]:

class CosmologyTestDataset(Dataset):
    def __init__(self, images: np.ndarray, mean: np.ndarray, std: np.ndarray):
        self.images = images.astype(np.float32)
        self.mean = mean.reshape(1, 1, 1).astype(np.float32)
        self.std = std.reshape(1, 1, 1).astype(np.float32)

    def __len__(self) -> int:
        return len(self.images)

    def __getitem__(self, idx: int) -> torch.Tensor:
        image = (self.images[idx] - self.mean) / self.std
        return torch.from_numpy(image).unsqueeze(0)


In [ ]:

def predict_with_uncertainty(model: nn.Module, dataloader: DataLoader, device: str) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    means, stds = [], []
    with torch.no_grad():
        for batch in dataloader:
            batch = batch.to(device)
            mean, log_var = model(batch)
            means.append(mean.cpu().numpy())
            stds.append(np.exp(0.5 * log_var.cpu().numpy()))
    return np.concatenate(means), np.concatenate(stds)


In [ ]:

test_dataset = CosmologyTestDataset(data_obj.kappa_test, image_mean, image_std)
test_loader = DataLoader(test_dataset, batch_size=config.VAL_BATCH_SIZE, shuffle=False)

test_means, test_stds = predict_with_uncertainty(trained_model, test_loader, config.DEVICE)
print(f"Test predictions shape: {test_means.shape}, Test std shape: {test_stds.shape}")


In [ ]:

submission_payload = {
    'means': test_means.tolist(),
    'errorbars': test_stds.tolist()
}

zip_path = Utility.save_json_zip(
    submission_dir=os.path.join(root_dir, 'submissions'),
    json_file_name='result.json',
    zip_file_name='phase1_advanced_submission.zip',
    data=submission_payload
)

print(f"Submission package created at: {zip_path}")


---
### 8 - Next steps

- **Hyperparameter sweeps:** Explore larger batch sizes (if GPU memory allows), different residual depths, or modern architectures (ConvNeXt, Swin) by swapping the backbone.
- **Simulation-based inference:** Replace the Gaussian head with a normalizing flow or ensemble Monte Carlo dropout to better capture non-Gaussian posteriors.
- **Domain adaptation:** Incorporate baryonic nuisance parameters as conditioning variables or adversarial penalties to further marginalize systematic effects.
- **Pipeline automation:** Convert this notebook to a script/notebook hybrid so you can run large sweeps on HPC clusters and track experiments with tools like Weights & Biases.

Armed with current literature insights and an extensible code base, you can iterate quickly toward a top-tier submission.
